# Whisper accent LoRA — procedure

Spike 01 of `research/`. **This notebook is checked in unexecuted.** It records the
procedure a real run would follow; the absence of outputs is the point. The claim, the
falsifier and the outcome live in `README.md` beside it.

Cells marked **light** run against `vmp` with the standard library only. Cells marked
**heavy** need `transformers`, `peft`, `datasets`, `torch` and a microphone.

| Step | Section |
|---|---|
| 1 | Corpus — the prompt cards |
| 2 | Record — audio capture and the manifest |
| 3 | Baseline — transcribe before touching the model |
| 4 | Fine-tune — LoRA on `q_proj` / `v_proj` |
| 5 | Eval — exact match and WER, train and held-out |
| 6 | Caveats — what this cannot show |

## 1. Corpus

The label is the sentence on the card, never a transcript. Sentences cover what the
deployed agent actually hears: commands, digit runs, proper nouns and place names, and
a few long-vowel minimal pairs where an accent moves the vowel.

Target size is 35 sentences, roughly two minutes of speech — the size the predecessor
used (see `README.md`, Outcome), not a size anything has justified. The held-out list
is recorded in the same session and never trained on; without it the falsifier cannot
be evaluated.

In [ ]:
# light
SENTENCES = [
    "Set a timer for fifteen minutes.",
    "Call Aoife at half past two.",
    "The order number is four seven two one.",
    "Confirm the delivery address in Cork.",
    "Remind me to ring Ciaran at nine.",
    # ... 35 in total: commands, digits, names, places, minimal pairs
]

HELD_OUT = [
    "Move the appointment to the first of October.",
    "Read back the reference code slowly.",
    "Who is joining the call from Gdansk?",
]

len(SENTENCES), len(HELD_OUT)

## 2. Record

One WAV per sentence, 16 kHz mono, one speaker, one room, one microphone. Keep the
conditions constant: this spike is about the speaker, and anything else that varies
between the two conditions ends up attributed to the adapter.

Manifest rows are the shape `vmp.training.whisper_lora.load_audio_manifest` validates:
`{"audio_path", "text", "speaker", "duration_s"}`.

In [ ]:
# heavy — needs a microphone (sounddevice) and soundfile
from pathlib import Path

import sounddevice as sd
import soundfile as sf

SR = 16_000
SPEAKER = "operator-01"
AUDIO = Path("data/accent/audio")
AUDIO.mkdir(parents=True, exist_ok=True)


def record(sentence: str, index: int, prefix: str = "train", seconds: float = 6.0) -> dict:
    print(f"[{prefix} {index:02d}] read aloud: {sentence}")
    buf = sd.rec(int(seconds * SR), samplerate=SR, channels=1)
    sd.wait()
    path = AUDIO / f"{SPEAKER}-{prefix}-{index:03d}.wav"
    sf.write(path, buf, SR)
    return {
        "audio_path": str(path),
        "text": sentence,            # the card, not a transcript
        "speaker": SPEAKER,
        "duration_s": round(sf.info(path).duration, 2),
    }


rows = [record(s, i) for i, s in enumerate(SENTENCES)]
held_out_rows = [record(s, i, prefix="heldout") for i, s in enumerate(HELD_OUT)]
held_out_paths = [r["audio_path"] for r in held_out_rows]

In [ ]:
# light — write and validate the manifest with the package helpers
import sys

sys.path.insert(0, "../../src")

from vmp.training.plan import write_jsonl
from vmp.training.whisper_lora import audio_manifest_stats, load_audio_manifest

write_jsonl("data/accent/accent_train.jsonl", rows)
write_jsonl("data/accent/accent_heldout.jsonl", held_out_rows)

audio_manifest_stats(load_audio_manifest("data/accent/accent_train.jsonl"))

## 3. Baseline

Transcribe every sentence with the unmodified model and keep the file. The comparison
is worthless if the baseline is re-run later with a different decoder, beam size or
model revision.

Note the framework split described in `README.md`: the fast inference path is MLX and
has no training API, so the baseline is taken with the same `transformers` pipeline the
training uses. An MLX baseline against a `transformers` adapted run would confound the
frameworks with the adapter.

In [ ]:
# heavy — transformers
import json

from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-small.en")

baseline = [asr(r["audio_path"])["text"].strip() for r in rows]
held_out_baseline = [asr(p)["text"].strip() for p in held_out_paths]

with open("data/accent/baseline.json", "w") as fh:
    json.dump({"train": baseline, "held_out": held_out_baseline}, fh, indent=2)

## 4. Fine-tune

LoRA on `q_proj` and `v_proj` in every attention block, `r=16`, `alpha=32`, `lr=1e-3`,
3 epochs. The plan is not written here — it is `configs/train_whisper_lora.toml`, loaded
as a `TrainingPlan`, so the notebook and the package train the same thing and the
`config_hash` is comparable afterwards.

Run the dry run first: it validates the manifest, hashes the data and prints the step
count, and imports nothing heavy.

In [ ]:
# light — the dry-run manifest
from vmp.training.plan import TrainingPlan
from vmp.training.whisper_lora import run_whisper_lora

plan = TrainingPlan.from_toml("../../configs/train_whisper_lora.toml")
plan.datasets = {"train": "data/accent/accent_train.jsonl"}

manifest = run_whisper_lora(plan, dry_run=True)
manifest["estimated_steps"], manifest["config_hash"], manifest["warnings"]

In [ ]:
# heavy — the real run: transformers Seq2SeqTrainer + peft, adapter written to output_dir
result = run_whisper_lora(plan, dry_run=False)
result["result"]

## 5. Eval

Two scorings, always reported together:

- **train set** — the 35 sentences the adapter saw. This is the number the predecessor
  reported. It shows the adaptation took, and nothing else.
- **held-out set** — sentences by the same speaker the adapter never saw. This is the
  number the falsifier is written against.

`exact_match` and `word_error_rate` come from `vmp.training.whisper_lora`, so the
notebook, the package and the release gate normalise text the same way.

In [ ]:
# heavy — transcribe with the adapter applied, same decoder settings as the baseline
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(plan.base_model, language="en", task="transcribe")
base = WhisperForConditionalGeneration.from_pretrained(plan.base_model)
adapter = PeftModel.from_pretrained(base, plan.output_dir).eval()


def transcribe_with_adapter(audio_path: str) -> str:
    import soundfile as sf

    audio, sr = sf.read(audio_path)
    features = processor(audio, sampling_rate=sr, return_tensors="pt").input_features
    ids = adapter.generate(features, num_beams=1, do_sample=False)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()


adapted = [transcribe_with_adapter(r["audio_path"]) for r in rows]
held_out_adapted = [transcribe_with_adapter(p) for p in held_out_paths]

In [ ]:
# light — scoring
from vmp.training.whisper_lora import exact_match, word_error_rate

references = [r["text"] for r in rows]

for split, refs, pair in (
    ("train", references, (("baseline", baseline), ("adapted", adapted))),
    ("held-out", HELD_OUT, (("baseline", held_out_baseline), ("adapted", held_out_adapted))),
):
    for name, hyps in pair:
        em = exact_match(refs, hyps)
        wer = word_error_rate(refs, hyps)
        print(f"{split:<9} {name:<9} {em.details['correct']}/{em.details['total']}"
              f"  wer {wer.metrics['wer']:.4f}")

## 6. Caveats

- **A train-set number is not a result.** The predecessor measured 11/35 -> 32/35 on the
  sentences it trained on, with no held-out set (alpha-core, cycle 5, 2026-09-12,
  `notebook_whisper_accent_lora.ipynb`). That is consistent with adaptation and equally
  consistent with memorising 35 strings. The held-out rows above are the ones that decide.
- **One speaker, one room, one microphone.** Nothing here generalises to the accent as a
  category, to a second speaker, or to a different channel.
- **No general-speech control.** An adapter can improve the target speaker by making the
  model worse at everyone else. A control set is needed before it goes near a shared
  deployment.
- **The adapter does not load into the MLX runtime.** PEFT writes a `transformers`
  adapter; serving it on the fast path needs a merge and a conversion, which is its own
  piece of work (`vmp.edge.export`) with its own numerical checks.
- **Decoder settings are part of the measurement.** Beam size, sampling and the
  language/task tokens must be identical across baseline and adapted runs, or the
  comparison measures the decoder.